# Mamba SOH — Long-sequence training (L=4096) — GH-10

Train `MambaSOHPredictor` ở chuỗi dài (warmup 256→4096 + gradient accumulation + attention pooling) trên Kaggle GPU.

**Trước khi chạy:**
1. Settings → Accelerator → **GPU P100/T4**
2. + Add Data → dataset NASA chứa `cleaned_dataset/metadata.csv` + `cleaned_dataset/data/*.csv`
3. (repo private) Add-ons → Secrets → tạo `GITHUB_TOKEN` = GitHub PAT

> Notebook này **KHÔNG sửa file** — chạy thẳng `preprocess_long.py` + `train.py --long` từ branch GH-10.
> Window=30 production giữ nguyên; artifact long lưu riêng `soh_mamba_long_v1.0.pth`.

## 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: bật GPU ở Settings -> Accelerator -> GPU P100/T4')

## 2 — Clone branch GH-10

In [ ]:
import subprocess
BRANCH  = 'feat/GH-10-mamba-long-seq-4096'
REPO    = '/kaggle/working/ai-module'
URL_PUB = 'https://github.com/GSU26SE55/ai-module.git'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{token}@github.com/GSU26SE55/ai-module.git'
except Exception as e:
    print('No GITHUB_TOKEN secret -> thử public clone:', e)
    url = URL_PUB
subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', url, REPO], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', URL_PUB])  # xoá token khỏi remote
print('Branch:', subprocess.check_output(['git','-C',REPO,'branch','--show-current']).decode().strip())
print('Commit:', subprocess.check_output(['git','-C',REPO,'log','-1','--oneline']).decode().strip())

## 3 — Dependencies (torch đã có sẵn trên Kaggle)

In [ ]:
%pip install -q scipy scikit-learn joblib pandas
import scipy, sklearn; print('scipy', scipy.__version__, '| sklearn', sklearn.__version__)

## 4 — Tìm NASA dataset + vào repo

In [ ]:
import os, subprocess
REPO = '/kaggle/working/ai-module'
found = [f for f in subprocess.check_output(['find','/kaggle/input','-name','metadata.csv']).decode().splitlines() if f]
assert found, 'Khong thay metadata.csv — + Add Data dataset NASA cleaned_dataset'
DATASET = os.path.dirname(found[0])
os.chdir(REPO)
print('DATASET    :', DATASET)
print('has data/  :', os.path.isdir(f'{DATASET}/data'))
print('cwd        :', os.getcwd())
print('scaler.pkl :', os.path.isfile('models/weights/scaler.pkl'), '(committed, preprocess_long reuse)')

## 5 — Preprocess (ghép cycle → chuỗi 4096)

Tạo `data/processed_long/{train,val,test}.pt` + `models/weights/feature_scaler_long.pkl`.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/preprocess_long.py --data-dir "{DATASET}" --output-dir data/processed_long

## 6 — Smoke test (1 epoch/stage) — kiểm tra pipeline trước

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/train.py --long --stage-epochs 1 --final-epochs 1 --micro-batch 8

## 7 — Full training

Warmup 256→512→1024→2048→4096 (early stopping ở stage cuối). GPU auto-detect, AMP fp16 tự bật.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/train.py --long

## 8 — Kết quả + đóng gói artifact

In [ ]:
import os, glob, shutil, torch
os.chdir('/kaggle/working/ai-module')
logs = sorted(glob.glob('logs/training/train_*.log'), key=os.path.getmtime)
if logs:
    print('Log:', logs[-1]); print('-'*50)
    !grep -E "Test MAE|Test RMSE|\[stage|Saved long" "{logs[-1]}"
ck = 'models/weights/soh_mamba_long_v1.0.pth'
if os.path.isfile(ck):
    c = torch.load(ck, map_location='cpu', weights_only=False)
    print('-'*50)
    print(f"seq_len={c['seq_len']} pooling={c['pooling']} | Test MAE={c['test_mae']}%  RMSE={c['test_rmse']}%")
# package 2 artifact de download + commit vao branch GH-10
os.makedirs('/kaggle/working/out', exist_ok=True)
for f in ['models/weights/soh_mamba_long_v1.0.pth', 'models/weights/feature_scaler_long.pkl']:
    if os.path.isfile(f): shutil.copy2(f, '/kaggle/working/out/'); print('copied', f)
shutil.make_archive('/kaggle/working/mamba_long_artifacts', 'zip', '/kaggle/working/out')
print('
Download: Output tab -> mamba_long_artifacts.zip')
print('Commit 2 file vao branch GH-10 (PR #11) + dien MAE vao PR.')

## Nếu MAE > 2% (khả năng cao do data-scarcity: 1083/159/50 windows)

Đây là trần data (3 pin NASA), không phải bug. Thử **hạ L=2048** rồi chạy lại Cell 5→7:
```python
# sửa LONG_SEQ_LEN = 2048 trong src/core/config.py
import re, pathlib
p = pathlib.Path('/kaggle/working/ai-module/src/core/config.py')
p.write_text(re.sub(r'LONG_SEQ_LEN\s*=\s*4096', 'LONG_SEQ_LEN    = 2048', p.read_text()))
```
